In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for Outbound FIA Table Schema and Data Update for DELETE_FLAG
# Purpose: Clone three Delta tables, rename 'Filler_1' to 'DELETE_FLAG', and enforce NOT NULL constraint on DELETE_FLAG
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script drops existing clone tables, reads source tables, filters out rows with NULL 'Filler_1', renames 'Filler_1' to 'DELETE_FLAG', creates clone tables with NOT NULL constraint on 'DELETE_FLAG', and inserts the filtered data.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import StringType, LongType, StructType, StructField  

CATALOG = "purgo_databricks"
SCHEMA = "purgo_playground"

# Table mapping: source table to clone table
TABLES = [
    {
        "source": "ods_free_drug",
        "clone": "ods_free_drug_clone",
        "columns": [
            ("PM_ID", StringType(), True),
            ("Shipment_Date", StringType(), True),
            ("Dispense_ID", LongType(), True),
            ("DELETE_FLAG", StringType(), False),
            ("Drug_Name", StringType(), True),
            ("Quantity", LongType(), True),
            ("Patient_ID", StringType(), True)
        ]
    },
    {
        "source": "ods_non_contracting",
        "clone": "ods_non_contracting_clone",
        "columns": [
            ("PM_ID", StringType(), True),
            ("Shipment_Date", StringType(), True),
            ("Dispense_ID", LongType(), True),
            ("DELETE_FLAG", StringType(), False),
            ("Facility", StringType(), True),
            ("Units_Provided", LongType(), True),
            ("Contract_Flag", StringType(), True)
        ]
    },
    {
        "source": "ods_copay",
        "clone": "ods_copay_clone",
        "columns": [
            ("PM_ID", StringType(), True),
            ("NDC", StringType(), True),
            ("Date_of_Service", StringType(), True),
            ("Rx_Claim_Number", StringType(), True),
            ("DELETE_FLAG", StringType(), False),
            ("Copay_Amount", LongType(), True),
            ("Plan_ID", StringType(), True)
        ]
    }
]

def drop_table_if_exists(table_full_name: str):
    """
    Drop the table if it exists.
    Args:
        table_full_name (str): Fully qualified table name
    Returns:
        None
    """
    # Purpose: Ensure clean slate for clone table creation
    spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")

def create_delta_table_with_not_null(table_full_name: str, columns: list):
    """
    Create a Delta table with specified columns, enforcing NOT NULL on DELETE_FLAG.
    Args:
        table_full_name (str): Fully qualified table name
        columns (list): List of (name, type, nullable) tuples
    Returns:
        None
    """
    # Purpose: DDL creation with NOT NULL constraint on DELETE_FLAG
    cols_ddl = []
    for name, dtype, nullable in columns:
        if isinstance(dtype, StringType):
            col_type = "STRING"
        elif isinstance(dtype, LongType):
            col_type = "BIGINT"
        else:
            raise Exception(f"Unsupported type for column {name}: {dtype}")
        null_str = "NOT NULL" if name == "DELETE_FLAG" else ""
        cols_ddl.append(f"{name} {col_type} {null_str}".strip())
    ddl = f"""
    CREATE TABLE {table_full_name} (
        {', '.join(cols_ddl)}
    ) USING DELTA
    """
    spark.sql(ddl)
    # Add NOT NULL constraint on DELETE_FLAG
    spark.sql(f"ALTER TABLE {table_full_name} ADD CONSTRAINT DELETE_FLAG_not_null CHECK (DELETE_FLAG IS NOT NULL)")

def clone_table_with_delete_flag(source_table: str, clone_table: str, columns: list):
    """
    Clone source table to clone table, renaming Filler_1 to DELETE_FLAG and enforcing NOT NULL.
    Args:
        source_table (str): Source table name (without catalog/schema)
        clone_table (str): Clone table name (without catalog/schema)
        columns (list): List of (name, type, nullable) tuples for clone table
    Returns:
        None
    """
    # Purpose: End-to-end clone logic for one table
    full_source = f"{CATALOG}.{SCHEMA}.{source_table}"
    full_clone = f"{CATALOG}.{SCHEMA}.{clone_table}"
    drop_table_if_exists(full_clone)
    try:
        df = spark.table(full_source)
    except Exception as e:
        raise Exception(f"Source table {source_table} does not exist")
    # Validate Filler_1 exists and is string
    src_schema = df.schema
    if "Filler_1" not in [f.name for f in src_schema.fields]:
        raise Exception(f"Source table {source_table} missing required column 'Filler_1'")
    for f in src_schema.fields:
        if f.name == "Filler_1" and not isinstance(f.dataType, StringType):
            raise Exception(f"Column 'Filler_1' in {source_table} must be of type string, found {f.dataType}")
    # Filter rows where Filler_1 IS NOT NULL
    df_filtered = df.filter(F.col("Filler_1").isNotNull())
    # Build select expressions: rename Filler_1 to DELETE_FLAG, keep column order
    select_exprs = []
    for name, dtype, nullable in columns:
        if name == "DELETE_FLAG":
            select_exprs.append(F.col("Filler_1").alias("DELETE_FLAG"))
        else:
            select_exprs.append(F.col(name))
    df_final = df_filtered.select(*select_exprs)
    # Create clone table with NOT NULL constraint
    create_delta_table_with_not_null(full_clone, columns)
    # Insert data
    df_final.write.format("delta").mode("append").saveAsTable(full_clone)

# Main execution: process all three tables
for tbl in TABLES:
    clone_table_with_delete_flag(tbl["source"], tbl["clone"], tbl["columns"])

# spark.stop()  # Do not stop SparkSession in Databricks
